# Computer Exercise 15.27 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.27 Sequential Decision Making — *Prioritized Replay + Component-Specific LR for +CNRT Rehabilitation*
> **풀이 일자**: Day 94
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 3.** Day 93 (§15.26 Problem 3) tested whether **uniform multi-slip curriculum** could
> rescue the +CNRT stack from the greedy-freeze negative finding of Day 91 P3, and found it
> could not (cross-slip mean 0.249 vs baseline 0.918). Attempt a different rehabilitation:
> **(a) prioritized replay** — sample transitions with probability proportional to $|\delta|^\alpha$
> where $\delta$ is TD error, $\alpha \in \{0.0, 0.5, 1.0\}$, and **(b) component-specific
> learning rate** — allow the Noisy head to use $\eta_N = k\eta,\ k \in \{1, 2, 5\}$ while
> the rest of +CNRT uses base $\eta = 0.05$. Train baseline vs +CNRT × 3 seeds × 800 steps and
> freeze-evaluate on three slip distributions $p_d \in \{0.05, 0.10, 0.20\}$. Report the
> cross-slip mean and generalization gap $\Gamma = \max_{p_d} R - \min_{p_d} R$, and check
> whether **any $(\alpha, k)$ combination reverses the Day 93 P3 negative finding**.

### 한국어 풀이용 정리
Day 91 P3 / Day 93 P3 의 +CNRT 열위를 uniform curriculum 이 아닌 **prioritized replay + Noisy 헤드
전용 higher lr** 처방으로 회복 가능한지 판정. $\alpha \in \{0.0, 0.5, 1.0\}$ × $k \in \{1, 2, 5\}$
격자에서 +CNRT 를 rehab 하고 baseline 과 cross-slip freeze 성능 비교.


## 2. 수학적 배경

### 2.1 Replay buffer & prioritization
버퍼 $B = \{(s_i, a_i, r_i, s'_i, d_i, \delta_i)\}$. 샘플 확률
$$
p_i = \frac{|\delta_i|^\alpha + \varepsilon}{\sum_j (|\delta_j|^\alpha + \varepsilon)}.
$$
$\alpha=0$ 이면 uniform, $\alpha=1$ 이면 fully proportional.

### 2.2 Component-specific lr
+CNRT 안에서 Noisy 헤드의 파라미터 $(W_N, b_N, \sigma_{W_N}, \sigma_{b_N})$ 는
$\eta_N = k\eta,\ k \in \{1, 2, 5\}$ 을 사용하고, 나머지 (트렁크, Cramér Twin heads, EMA) 는
$\eta = 0.05$.

### 2.3 Freeze evaluation
훈련 시 $p_{\text{train}}=0.10$. Freeze 후 3 배포 슬립 $p_d \in \{0.05, 0.10, 0.20\}$ 에서 각
60 에피소드 greedy. Cross-slip mean 과 $\Gamma = R_{\max} - R_{\min}$.


## 3. 풀이 흐름

1. Chain MDP + replay buffer.
2. `PlainAgent` (baseline) 와 `CNRTAgent` (Noisy + Cramér Twin + EMA, component-lr split).
3. 격자: $\alpha \in \{0, 0.5, 1\}$ × $k \in \{1, 2, 5\}$ — +CNRT 만. baseline 은 uniform ($\alpha=0$).
4. 3 시드 × 800 step 학습.
5. Freeze 후 3 slip 배포 각 60 에피소드 greedy → cross-slip mean, $\Gamma$.
6. Heatmap: cross-slip mean over $(\alpha, k)$ + comparison bar (best +CNRT vs baseline).
7. 결론.


In [1]:

import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N, self.p_slip, self.step_r, self.goal_r = N, p_slip, step_r, goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip:
            a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def one_hot(s, N):
    x = np.zeros(N); x[s] = 1.0; return x


class ReplayBuffer:
    def __init__(self, cap=2000, rng=None):
        self.cap = cap; self.rng = rng or np.random.default_rng(0)
        self.data = []; self.delta = []
    def push(self, tr, d=1.0):
        self.data.append(tr); self.delta.append(abs(d))
        if len(self.data) > self.cap:
            self.data.pop(0); self.delta.pop(0)
    def sample(self, alpha=0.0, eps=1e-3):
        n = len(self.data)
        if n == 0: return None, None
        w = (np.array(self.delta) ** alpha) + eps
        p = w / w.sum()
        idx = int(self.rng.choice(n, p=p))
        return self.data[idx], idx
    def update_delta(self, idx, d):
        self.delta[idx] = abs(d)


class PlainAgent:
    def __init__(self, seed=0, H=16, N=5, A=2, eta=0.05, gamma=0.95, eps=0.15):
        self.rng = np.random.default_rng(seed); self.H, self.N, self.A = H, N, A
        self.eta = eta; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, 0.3, size=(N, H)); self.b1 = np.zeros(H)
        self.W2 = [self.rng.normal(0, 0.3, size=(H, 1)) for _ in range(A)]
        self.b2 = [np.zeros(1) for _ in range(A)]
    def _phi(self, s):
        x = one_hot(s, self.N); h = np.tanh(self.W1.T @ x + self.b1); return x, h
    def Q(self, s):
        _, h = self._phi(s)
        return np.array([float((h @ self.W2[a] + self.b2[a])[0]) for a in range(self.A)])
    def act(self, s, greedy=False):
        if not greedy and self.rng.random() < self.eps: return int(self.rng.integers(self.A))
        return int(np.argmax(self.Q(s)))
    def update(self, tr):
        s, a, r, sp, d = tr
        x, h = self._phi(s)
        q = float((h @ self.W2[a] + self.b2[a])[0])
        target = r if d else r + self.gamma * float(self.Q(sp).max())
        err = q - target
        grad_out = np.array([err])
        self.W2[a] -= self.eta * np.outer(h, grad_out); self.b2[a] -= self.eta * grad_out
        grad_h = self.W2[a] @ grad_out
        gd = grad_h * (1 - h**2)
        self.W1 -= self.eta * np.outer(x, gd); self.b1 -= self.eta * gd
        return err


class CNRTAgent:
    def __init__(self, seed=0, H=16, N=5, A=2, eta=0.05, k=1.0, gamma=0.95, eps=0.15):
        self.rng = np.random.default_rng(seed); self.H, self.N, self.A = H, N, A
        self.eta = eta; self.eta_N = k * eta; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, 0.3, size=(N, H)); self.b1 = np.zeros(H)
        self.KA, self.KB = 5, 5
        self.W2A = [self.rng.normal(0, 0.3, size=(H, self.KA)) for _ in range(A)]
        self.b2A = [np.zeros(self.KA) for _ in range(A)]
        self.W2B = [self.rng.normal(0, 0.3, size=(H, self.KB)) for _ in range(A)]
        self.b2B = [np.zeros(self.KB) for _ in range(A)]
        self.WN = [self.rng.normal(0, 0.3, size=(H, 1)) for _ in range(A)]
        self.bN = [np.zeros(1) for _ in range(A)]
        self.sWN = [np.full((H, 1), 0.1) for _ in range(A)]
        self.sbN = [np.full(1, 0.1) for _ in range(A)]
        self.mu = np.zeros(H); self.var = np.ones(H); self.beta_ema = 0.99
        self.atomsA = np.linspace(-1.0, 1.0, self.KA)
        self.atomsB = np.linspace(-1.0, 1.0, self.KB)
    def _phi(self, s, update_ema=False):
        x = one_hot(s, self.N)
        h_raw = np.tanh(self.W1.T @ x + self.b1)
        if update_ema:
            self.mu = self.beta_ema * self.mu + (1 - self.beta_ema) * h_raw
            self.var = self.beta_ema * self.var + (1 - self.beta_ema) * (h_raw - self.mu) ** 2
        h = (h_raw - self.mu) / np.sqrt(self.var + 1e-6)
        return x, h_raw, h
    def _q(self, h, a):
        lA = h @ self.W2A[a] + self.b2A[a]; pA = np.exp(lA - lA.max()); pA/=pA.sum()
        lB = h @ self.W2B[a] + self.b2B[a]; pB = np.exp(lB - lB.max()); pB/=pB.sum()
        qA = float((pA * self.atomsA).sum()); qB = float((pB * self.atomsB).sum())
        qN = float((h @ self.WN[a] + self.bN[a])[0])
        return (qA + qB + qN) / 3.0
    def Q(self, s):
        _, _, h = self._phi(s)
        return np.array([self._q(h, a) for a in range(self.A)])
    def act(self, s, greedy=False):
        if not greedy and self.rng.random() < self.eps: return int(self.rng.integers(self.A))
        return int(np.argmax(self.Q(s)))
    def _proj(self, target, atoms):
        K = len(atoms)
        t = np.clip(target, atoms[0], atoms[-1])
        idx = np.searchsorted(atoms, t); idx = max(1, min(K - 1, idx))
        lo, hi = atoms[idx-1], atoms[idx]
        w_hi = (t - lo) / (hi - lo + 1e-12)
        tgt = np.zeros(K); tgt[idx-1] = 1 - w_hi; tgt[idx] = w_hi
        return tgt
    def update(self, tr):
        s, a, r, sp, d = tr
        x, h_raw, h = self._phi(s, update_ema=True)
        target = r if d else r + self.gamma * float(self.Q(sp).max())
        gW1_agg = np.zeros_like(self.W1); gb1_agg = np.zeros_like(self.b1)
        # Twin A
        z = h @ self.W2A[a] + self.b2A[a]; m = np.exp(z - z.max()); m/=m.sum()
        tgt = self._proj(target, self.atomsA); grad_z = m - tgt
        self.W2A[a] -= self.eta * np.outer(h, grad_z); self.b2A[a] -= self.eta * grad_z
        grad_h_A = self.W2A[a] @ grad_z
        d1 = grad_h_A * (1 - h_raw**2); gW1_agg += np.outer(x, d1); gb1_agg += d1
        # Twin B
        z = h @ self.W2B[a] + self.b2B[a]; m = np.exp(z - z.max()); m/=m.sum()
        tgt = self._proj(target, self.atomsB); grad_z = m - tgt
        self.W2B[a] -= self.eta * np.outer(h, grad_z); self.b2B[a] -= self.eta * grad_z
        grad_h_B = self.W2B[a] @ grad_z
        d1 = grad_h_B * (1 - h_raw**2); gW1_agg += np.outer(x, d1); gb1_agg += d1
        # Noisy scalar head
        eps_w = self.rng.standard_normal(self.WN[a].shape)
        eps_b = self.rng.standard_normal(self.bN[a].shape)
        W = self.WN[a] + self.sWN[a] * eps_w
        b = self.bN[a] + self.sbN[a] * eps_b
        qN = float((h @ W + b)[0]); errN = qN - target
        grad_out = np.array([errN])
        self.WN[a] -= self.eta_N * np.outer(h, grad_out); self.bN[a] -= self.eta_N * grad_out
        self.sWN[a] -= self.eta_N * np.outer(h, grad_out) * eps_w
        self.sbN[a] -= self.eta_N * grad_out * eps_b
        grad_h_N = W @ grad_out
        d1 = grad_h_N * (1 - h_raw**2); gW1_agg += np.outer(x, d1); gb1_agg += d1
        self.W1 -= self.eta * gW1_agg; self.b1 -= self.eta * gb1_agg
        # report combined TD-err for priority
        _, _, h2 = self._phi(s)
        return self._q(h2, a) - target


def train_agent(kind, seed, alpha=0.0, k=1.0, T=800, buffer_cap=2000):
    env = ChainMDP(rng=np.random.default_rng(seed + 7))
    buf = ReplayBuffer(cap=buffer_cap, rng=np.random.default_rng(seed + 13))
    ag = PlainAgent(seed=seed) if kind == 'plain' else CNRTAgent(seed=seed, k=k)
    s = env.reset()
    for _ in range(T):
        a = ag.act(s, greedy=False)
        sp, r, done = env.step(a)
        buf.push((s, a, r, sp, done), d=1.0)
        tr, idx = buf.sample(alpha=alpha)
        if tr is not None:
            err = ag.update(tr)
            buf.update_delta(idx, err)
        s = env.reset() if done else sp
    return ag


def freeze_eval(agent, p_slip, seed, n_eval=60, max_len=50):
    env = ChainMDP(p_slip=p_slip, rng=np.random.default_rng(seed + 5000))
    rets = []
    for ep in range(n_eval):
        env.rng = np.random.default_rng(seed + 5000 + ep)
        s = env.reset(); total = 0.0
        for _ in range(max_len):
            a = agent.act(s, greedy=True)
            s, r, done = env.step(a); total += r
            if done: break
        rets.append(total)
    return float(np.mean(rets))
print("PR + CNRT agent ready.")


/tmp/mplcfg is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-2h180rj9 because there was an issue with the default path (/tmp/mplcfg); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


PR + CNRT agent ready.


In [2]:

SEEDS = [94301, 94302, 94303]
SLIPS = [0.05, 0.10, 0.20]
ALPHAS = [0.0, 0.5, 1.0]
KS = [1.0, 2.0, 5.0]

rows = []
for sd in SEEDS:
    ag = train_agent('plain', sd, alpha=0.0, k=1.0)
    per = [freeze_eval(ag, p, sd) for p in SLIPS]
    rows.append({'agent':'baseline', 'alpha': 0.0, 'k': 1.0, 'seed': sd,
                 'R_p05': per[0], 'R_p10': per[1], 'R_p20': per[2]})

for alpha in ALPHAS:
    for k in KS:
        for sd in SEEDS:
            ag = train_agent('cnrt', sd, alpha=alpha, k=k)
            per = [freeze_eval(ag, p, sd) for p in SLIPS]
            rows.append({'agent':'+CNRT', 'alpha': alpha, 'k': k, 'seed': sd,
                         'R_p05': per[0], 'R_p10': per[1], 'R_p20': per[2]})

df3 = pd.DataFrame(rows)
df3['cross_mean'] = df3[['R_p05','R_p10','R_p20']].mean(axis=1)
df3['Gamma'] = df3[['R_p05','R_p10','R_p20']].max(axis=1) - df3[['R_p05','R_p10','R_p20']].min(axis=1)
df3


,agent,alpha,k,seed,R_p05,R_p10,R_p20,cross_mean,Gamma
0,baseline,0.0000,1.0000,94301,0.9313,0.9237,0.9013,0.9188,0.0300
1,baseline,0.0000,1.0000,94302,0.9313,0.9237,0.9003,0.9184,0.0310
2,baseline,0.0000,1.0000,94303,0.9313,0.9237,0.9017,0.9189,0.0297
3,+CNRT,0.0000,1.0000,94301,0.9313,0.9237,0.9013,0.9188,0.0300
4,+CNRT,0.0000,1.0000,94302,0.9313,0.9237,0.9003,0.9184,0.0310
5,+CNRT,0.0000,1.0000,94303,0.9313,0.9237,0.9017,0.9189,0.0297
6,+CNRT,0.0000,2.0000,94301,0.9313,0.9237,0.9013,0.9188,0.0300
7,+CNRT,0.0000,2.0000,94302,0.9313,0.9237,0.9003,0.9184,0.0310
8,+CNRT,0.0000,2.0000,94303,0.9313,0.9237,0.9017,0.9189,0.0297
9,+CNRT,0.0000,5.0000,94301,0.9313,0.9237,0.9013,0.9188,0.0300


In [3]:

agg = df3.groupby(['agent','alpha','k']).agg(
    cross_mean=('cross_mean', 'mean'),
    cross_std=('cross_mean', 'std'),
    Gamma=('Gamma', 'mean'),
).reset_index()
agg


,agent,alpha,k,cross_mean,cross_std,Gamma
0,+CNRT,0.0000,1.0000,0.9187,0.0002,0.0302
1,+CNRT,0.0000,2.0000,0.9187,0.0002,0.0302
2,+CNRT,0.0000,5.0000,-0.3291,1.0807,0.0896
3,+CNRT,0.5000,1.0000,0.7249,0.3359,0.2340
4,+CNRT,0.5000,2.0000,0.7259,0.3337,0.2340
5,+CNRT,0.5000,5.0000,0.2949,1.0807,0.0597
6,+CNRT,1.0000,1.0000,0.9187,0.0002,0.0302
7,+CNRT,1.0000,2.0000,-0.3291,1.0807,0.0896
8,+CNRT,1.0000,5.0000,-0.2330,1.0076,0.2622
9,baseline,0.0000,1.0000,0.9187,0.0002,0.0302


In [4]:

cnrt = agg[agg.agent=='+CNRT']
mat = cnrt.pivot(index='alpha', columns='k', values='cross_mean')
gmat = cnrt.pivot(index='alpha', columns='k', values='Gamma')

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
im0 = axes[0].imshow(mat.values, cmap='viridis', aspect='auto')
axes[0].set_xticks(range(len(KS))); axes[0].set_xticklabels([f'k={k}' for k in KS])
axes[0].set_yticks(range(len(ALPHAS))); axes[0].set_yticklabels([f'a={a}' for a in ALPHAS])
axes[0].set_title('+CNRT cross-slip mean')
for i in range(len(ALPHAS)):
    for j in range(len(KS)):
        v = mat.values[i,j]
        axes[0].text(j, i, f'{v:.3f}', ha='center', va='center',
                     color='white' if v < 0.4 else 'black', fontsize=9)
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(gmat.values, cmap='magma', aspect='auto')
axes[1].set_xticks(range(len(KS))); axes[1].set_xticklabels([f'k={k}' for k in KS])
axes[1].set_yticks(range(len(ALPHAS))); axes[1].set_yticklabels([f'a={a}' for a in ALPHAS])
axes[1].set_title('+CNRT generalization gap Gamma')
for i in range(len(ALPHAS)):
    for j in range(len(KS)):
        v = gmat.values[i,j]
        axes[1].text(j, i, f'{v:.3f}', ha='center', va='center',
                     color='white' if v > 0.4 else 'black', fontsize=9)
fig.colorbar(im1, ax=axes[1])
fig.suptitle('Day 94 P3 — +CNRT rehab via prioritized replay + Noisy lr', y=1.05)
plt.savefig('/tmp/day94_p3_heat.png', dpi=90, bbox_inches='tight'); plt.show()


In [5]:

best_row = cnrt.loc[cnrt.cross_mean.idxmax()]
base_per = df3[df3.agent=='baseline'][['R_p05','R_p10','R_p20']].mean().values
mask = (df3.agent=='+CNRT') & (df3.alpha==best_row.alpha) & (df3.k==best_row.k)
best_per = df3[mask][['R_p05','R_p10','R_p20']].mean().values

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3); w = 0.35
ax.bar(x - w/2, base_per, width=w, label='baseline (alpha=0, k=1)', color='#888')
ax.bar(x + w/2, best_per, width=w,
       label=f'+CNRT best (alpha={best_row.alpha}, k={best_row.k})', color='#c14')
ax.set_xticks(x); ax.set_xticklabels([f'p_d={p}' for p in SLIPS])
ax.set_ylabel('freeze greedy return')
ax.set_title('Day 94 P3 — Cross-slip: baseline vs best +CNRT')
ax.legend(); plt.tight_layout()
plt.savefig('/tmp/day94_p3_bar.png', dpi=90, bbox_inches='tight'); plt.show()
print('best +CNRT: alpha={}, k={}, cross_mean={:.3f}, Gamma={:.3f}'.format(
    best_row.alpha, best_row.k, best_row.cross_mean, best_row.Gamma))
print('baseline cross_mean = {:.3f}'.format(base_per.mean()))


best +CNRT: alpha=0.0, k=1.0, cross_mean=0.919, Gamma=0.030
baseline cross_mean = 0.919


## 4. 결과 해석

1. **격자에서 최적 셀** — $(\alpha, k)$ 상 +CNRT 의 cross-slip mean 이 최대가 되는 셀에서
   baseline 과의 gap $\Delta = R^{+\text{CNRT}}_{\text{best}} - R^{\text{base}}$ 을 확인. Day 93 P3
   default 대비 개선 폭이 곧 rehabilitation 의 크기.
2. **$\Gamma$ (generalization gap)** — 슬립 배포 변화에 대한 민감도. Noisy 헤드의 higher lr
   이 $\Gamma$ 를 축소시키는지 (더 robust) 여부.
3. **Prioritized replay 의 역할** — $\alpha$ 를 키우면 high-TD 전이에 집중해 정확한 값 함수를
   먼저 수렴시키지만, +CNRT 처럼 이미 noisy gradient 를 가진 처방에는 오히려 진동을 증폭시킬
   수 있음. 실제로 어느 방향이 이겼는지는 위 heatmap 이 판정.

> **결론**: prioritized replay + Noisy-only higher lr 은 Day 93 P3 uniform curriculum 처방과 다른
> 축으로 +CNRT 를 개선하는지, 그리고 baseline 을 회복하는지가 격자 결과가 말해준다. 최적 셀의
> $\Delta$ 부호가 최종 판정.

다음 Day (§15.28) 에서는 (a) 격자에서 남은 미해결 원인 (특히 EMA whitening warmup) 을 별도
gate 로 처방화하고, (b) chain MDP 이외의 4-arm bandit 확장에서 재현성을 검증할 예정.
